In [ ]:
import json
import re
import requests
from bs4 import BeautifulSoup
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =====================================
# CONFIG
# =====================================

URL_TABLA3 = "https://www.fao.org/gsfaonline/reference/table3.html"

HEADERS = {"User-Agent": "Mozilla/5.0"}

ARCHIVO_SALIDA = "tabla3.json"


# =====================================
# EXTRAER INS DE GRUPO
# =====================================

def extraer_ins_grupo(url_grupo):

    r = requests.get(url_grupo + "&lang=en", headers=HEADERS, timeout=30, verify=False)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    tablas = soup.find_all("table")

    tabla_objetivo = None
    for i, tabla in enumerate(tablas):
        texto = tabla.get_text(" ", strip=True).strip()
        if texto == "INS No.":
            if i + 1 < len(tablas):
                tabla_objetivo = tablas[i + 1]
            break

    if tabla_objetivo is None:
        print(f"  ⚠️ No se encontró tabla en grupo: {url_grupo}")
        return []

    ins_lista = []
    for tr in tabla_objetivo.find_all("tr"):
        celdas = tr.find_all("td")
        if len(celdas) < 3:
            continue
        ins = celdas[1].get_text(strip=True)
        if not ins:
            continue
        if not any(c.isdigit() for c in ins):
            continue
        if len(ins) > 20:
            continue
        ins_lista.append(ins)

    return ins_lista


# =====================================
# SCRAPING TABLA 3
# =====================================

print("🔍 Descargando tabla 3...")

r = requests.get(URL_TABLA3, headers=HEADERS, timeout=30, verify=False)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

ins_lista = []

for a in soup.find_all("a", href=True):

    href = a["href"]
    texto = a.get_text(strip=True)

    # -------------------------
    # GRUPO → ir a buscar sus INS
    # -------------------------
    if "groups/details" in href:
        print(f"  🔗 Grupo encontrado: {texto} → {href}")
        url_grupo = href if href.startswith("http") else "https://www.fao.org" + href
        ins_grupo = extraer_ins_grupo(url_grupo)
        print(f"     ✅ {len(ins_grupo)} INS extraídos: {ins_grupo}")
        for ins in ins_grupo:
            if ins not in ins_lista:
                ins_lista.append(ins)
        continue

    # -------------------------
    # ADITIVO INDIVIDUAL → extraer INS del texto
    # -------------------------
    if "additives/details" not in href:
        continue

    # Extraer contenido entre el primer ( externo y el último )
    # para capturar correctamente INS como 503(i), 1100(iv), etc.
    primer_abre = texto.find("(")
    ultimo_cierre = texto.rfind(")")

    if primer_abre == -1 or ultimo_cierre == -1:
        continue

    ins = texto[primer_abre + 1:ultimo_cierre].strip()

    if ins not in ins_lista:
        ins_lista.append(ins)

# =====================================
# GUARDAR
# =====================================

with open(ARCHIVO_SALIDA, "w", encoding="utf-8") as f:
    json.dump(ins_lista, f, ensure_ascii=False, indent=4)

print(f"\n✅ Total INS extraídos: {len(ins_lista)}")
print(f"💾 Guardado en: {ARCHIVO_SALIDA}")

🔍 Descargando tabla 3...
  🔗 Grupo encontrado: RIBOFLAVINS → /gsfaonline/groups/details.html?id=96
     ✅ 4 INS extraídos: ['101(ii)', '101(iv)', '101(iii)', '101(i)']

✅ Total INS extraídos: 138
💾 Guardado en: tabla3.json
